In [ ]:
import geemap
import ee
import pandas as pd
import geopandas as gpd
import rasterio as rio

# Authenticate once in this machine/session before running the notebook end-to-end.
ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com',
)

# Set the path to the JSON file containing the geometry.
path_json = "../data/bacias_meso_SF.geojson"
gdf = gpd.read_file(path_json)
gdf_menor = gdf[gdf['nm_mesoRH'] == 'Baixo São Francisco']
geom_ee = geemap.geopandas_to_ee(gdf_menor)
area = geom_ee.geometry()

# 3. Carregar a coleção MODIS (Reflectância de Superfície)
modis = ee.ImageCollection('MODIS/061/MOD09A1') \
    .filterBounds(area) \
    .filterDate('2000-02-18', '2025-12-31')

In [ ]:
# Função para calcular o BSI (Bare Soil Index)
def bsi_index(image):
    # Aplicar fator de escala do MODIS para obter a reflectância real
    img_scaled = image.multiply(0.0001)

    # Cálculo do BSI usando .expression()
    # O BSI realça o solo exposto ao contrastar o SWIR/RED com o NIR/BLUE
    bsi = img_scaled.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))', {
            'BLUE': img_scaled.select('sur_refl_b03'), # Banda 3 do MODIS (Azul)
            'RED':  img_scaled.select('sur_refl_b01'), # Banda 1 do MODIS (Vermelho)
            'NIR':  img_scaled.select('sur_refl_b02'), # Banda 2 do MODIS (Infravermelho Próximo)
            'SWIR': img_scaled.select('sur_refl_b06')  # Banda 6 do MODIS (Infravermelho de Ondas Curtas)
        }
    ).rename('BSI')

    # Adiciona a banda BSI à imagem original e mantém as propriedades de data
    return image.addBands(bsi).copyProperties(image, ['system:time_start'])




In [ ]:
# Carregar dataset com os indices de desertificação